# 00 — Data Audit: NFL Team Season Win Totals

**The gate.** Do usable point-in-time preseason win-total lines exist? `01`–`05` may not run until
this says so. Nothing is fitted here — only measurement.

**Reads:** `nflreadpy` schedules (or the pinned snapshot) + `futures/data/win_totals.csv`.
**Writes:** `futures/artifacts/data_audit.json` — verdict, frozen fold sets, hashes, provenance.

**Verdicts:** `GO` (needs a named book) · `GO-TIER-B` (§10 Amendment 1: archived consensus, `book`
null → §7 gates A+B only, gate C stays shut) · `NO-GO` (stop; substituting or reconstructing lines
is forbidden).

```bash
papermill futures/season_team_totals/00_data_audit.ipynb /tmp/out.ipynb
papermill futures/season_team_totals/00_data_audit.ipynb /tmp/out.ipynb -p OFFLINE True
papermill futures/season_team_totals/00_data_audit.ipynb /tmp/out.ipynb -p TIER_B_ARCHIVE False  # frozen gate
```

## Section 1 — Parameters

Papermill-overridable config in one tagged cell. `SEASON_MIN=2002` is the current 32-team alignment;
`SEASON_MAX`/`TARGET_SEASON` resolve from the data in Section 3 so the notebook doesn't go stale.
`TIER_B_ARCHIVE` toggles §10 Amendment 1 (False = frozen pre-amendment gate).

In [1]:
SEASON_MIN     = 2002          # first season considered (2002 = current 32-team alignment)
SEASON_MAX     = None          # None = latest season with completed regular-season games
TARGET_SEASON  = None          # None = latest season with a published schedule (the predict season)
LINES_PATH     = None          # None -> $FUTURES_LINES_PATH -> futures/data/win_totals.csv
OFFLINE        = None          # None -> $APP_OFFLINE; True forbids network (snapshot required)
WRITE_ARTIFACTS = True         # write artifacts/data_audit.json (+ schedule snapshot on a live pull)
TIER_B_ARCHIVE = True          # PREREGISTRATION §10 Amendment 1 (accepted 2026-08-03): admit an
                               # archived market consensus with a null `book` for Tier B ONLY.
                               # False reproduces the frozen pre-amendment gate exactly.
SEED           = 20260802      # pinned; no stochastic step here, pinned for provenance parity
RUN_TESTS      = True

### Interpreting the output

Prints nothing — it only binds names, and papermill replaces the whole cell at run time. The values
that actually ran are echoed by the test below and stamped into `data_audit.json`.

### What these tests guard

Type/range checks before anything expensive, because papermill injects whatever the caller typed.
An inverted window would otherwise produce an empty audit that "passes" by having nothing to check.

In [2]:
if RUN_TESTS:
    assert isinstance(SEASON_MIN, int) and SEASON_MIN >= 1999, "SEASON_MIN must be >= 1999 (nflverse schedule coverage)"
    assert SEASON_MAX is None or (isinstance(SEASON_MAX, int) and SEASON_MAX >= SEASON_MIN)
    assert TARGET_SEASON is None or isinstance(TARGET_SEASON, int)
    assert isinstance(SEED, int)
    assert isinstance(TIER_B_ARCHIVE, bool), "TIER_B_ARCHIVE gates a preregistered amendment — must be an explicit bool"
    print(f"✓ Section 1 tests passed | SEASON_MIN={SEASON_MIN} SEASON_MAX={SEASON_MAX} "
          f"TARGET_SEASON={TARGET_SEASON} SEED={SEED} TIER_B_ARCHIVE={TIER_B_ARCHIVE}")

✓ Section 1 tests passed | SEASON_MIN=2002 SEASON_MAX=None TARGET_SEASON=None SEED=20260802 TIER_B_ARCHIVE=True


### Reading the test result

The run's configuration receipt. If it disagrees with what you passed, stop here.
Does **not** prove the window is *sensible* — only that it is well-formed.

## Section 2 — Imports, paths, provenance

Walks up for the repo root (`app.py` + `futures/`), so this runs from anywhere. `_rel()` reports
repo-relative inside the repo, absolute outside. Provenance is stamped now, not at the end — an
artifact without library versions, as-of date and seed is a number without an origin.

In [3]:
import hashlib
import json
import os
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def _find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "app.py").exists() and (p / "futures").is_dir():
            return p
    raise RuntimeError(f"repo root not found above {start} (looked for app.py + futures/)")


REPO      = _find_repo_root(Path.cwd())
FUTURES   = REPO / "futures"
DATA_DIR  = FUTURES / "data"
ART_DIR   = FUTURES / "artifacts"
for _d in (DATA_DIR, ART_DIR):
    _d.mkdir(parents=True, exist_ok=True)

SNAPSHOT_PATH = DATA_DIR / "schedules_snapshot.parquet"
AUDIT_PATH    = ART_DIR / "data_audit.json"

# OFFLINE: explicit parameter wins, else the site-wide APP_OFFLINE switch.
if OFFLINE is None:
    OFFLINE = os.environ.get("APP_OFFLINE", "") == "1"
OFFLINE = bool(OFFLINE)

# Line file: explicit parameter -> env -> conventional location.
_lines_candidates = [
    ("parameter LINES_PATH", Path(LINES_PATH) if LINES_PATH else None),
    ("env FUTURES_LINES_PATH", Path(os.environ["FUTURES_LINES_PATH"]) if os.environ.get("FUTURES_LINES_PATH") else None),
    ("futures/data/win_totals.csv", DATA_DIR / "win_totals.csv"),
]
_lines_candidates = [(w, (p if p.is_absolute() else REPO / p)) for w, p in _lines_candidates if p is not None]

RUN_AT = datetime.now(timezone.utc)


def _rel(path: Path) -> str:
    """Repo-relative posix path when the file lives inside the repo, else the absolute
    path — a line file supplied via $FUTURES_LINES_PATH may legitimately sit outside it."""
    path = Path(path)
    try:
        return path.resolve().relative_to(REPO).as_posix()
    except ValueError:
        return str(path.resolve())


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_frame(df: pd.DataFrame) -> str:
    return hashlib.sha256(
        pd.util.hash_pandas_object(df.reset_index(drop=True), index=False).values.tobytes()
    ).hexdigest()


PROVENANCE = {
    "notebook": "futures/season_team_totals/00_data_audit.ipynb",
    "run_at_utc": RUN_AT.isoformat(),
    "as_of_date": RUN_AT.date().isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "seed": SEED,
    "offline": OFFLINE,
    "repo_root": str(REPO),
}
print(f"repo={REPO}\noffline={OFFLINE}  as_of={PROVENANCE['as_of_date']}")

repo=C:\Users\josep\Desktop\random_stuff\cowork_OS\JoSchoAnalytics
offline=False  as_of=2026-08-03


### Interpreting the output

Repo root should be `…/JoSchoAnalytics` regardless of where the kernel started. `as_of` is the date
to judge staleness by. `sha256_file` pins an input file; `sha256_frame` catches an upstream revision
that leaves the filename unchanged.

### What these tests guard

Root resolution worked, output dirs exist, and the hashers are both **deterministic and
content-sensitive** — a hash that never changes advertises provenance it isn't providing.

In [4]:
if RUN_TESTS:
    assert (REPO / "app.py").exists(), "repo root resolution failed"
    assert DATA_DIR.is_dir() and ART_DIR.is_dir(), "futures data/artifacts dirs not created"
    assert PROVENANCE["as_of_date"] == RUN_AT.date().isoformat()
    # the hashers must be deterministic and content-sensitive
    _a = pd.DataFrame({"x": [1, 2, 3]})
    _b = pd.DataFrame({"x": [1, 2, 4]})
    assert sha256_frame(_a) == sha256_frame(_a.copy()), "frame hash must be deterministic"
    assert sha256_frame(_a) != sha256_frame(_b), "frame hash must be content-sensitive"
    assert _rel(DATA_DIR / "x.csv") == "futures/data/x.csv", "in-repo paths must report repo-relative"
    assert Path(_rel(Path.home() / "outside.csv")).is_absolute(), "out-of-repo paths must stay absolute"
    print(f"✓ Section 2 tests passed | repo={REPO.name} offline={OFFLINE} "
          f"line-source candidates={len(_lines_candidates)}")

✓ Section 2 tests passed | repo=JoSchoAnalytics offline=False line-source candidates=1


### Reading the test result

Repo name, offline state, and how many line-source candidates will be searched. `1` = only the
conventional path; `2–3` = a caller supplied one. Does **not** prove any of them is usable.

## Section 3 — Load schedules (snapshot-aware)

Source for both the outcome and the season structure (first kickoff per season, games scheduled).
Prefers a pinned parquet snapshot, refuses to run offline without one. Franchises normalize to
current abbreviations (`OAK→LV`, `SD→LAC`, `STL→LA`) so a team joins its line across a relocation.

In [5]:
FRANCHISE_MAP = {"OAK": "LV", "SD": "LAC", "STL": "LA"}

if SNAPSHOT_PATH.exists():
    sched_raw = pd.read_parquet(SNAPSHOT_PATH)
    SCHED_SOURCE = f"snapshot:{_rel(SNAPSHOT_PATH)}"
    SCHED_LIB_VERSION = None
elif OFFLINE:
    raise RuntimeError(
        f"OFFLINE run with no schedule snapshot at {SNAPSHOT_PATH}. "
        "Run this notebook once online (OFFLINE=False) to create it."
    )
else:
    import nflreadpy as nfl
    sched_raw = nfl.load_schedules().to_pandas()
    SCHED_LIB_VERSION = getattr(nfl, "__version__", "unknown")
    SCHED_SOURCE = f"nflreadpy=={SCHED_LIB_VERSION} live pull"
    if WRITE_ARTIFACTS:
        sched_raw.to_parquet(SNAPSHOT_PATH, index=False)

sched = sched_raw[sched_raw["game_type"] == "REG"].copy()
sched["gameday"] = pd.to_datetime(sched["gameday"], errors="coerce")
for _side in ("home", "away"):
    sched[f"{_side}_franchise"] = sched[f"{_side}_team"].replace(FRANCHISE_MAP)

# Season window: SEASON_MAX defaults to the latest season with completed games; TARGET_SEASON to
# the latest season with a published schedule (which may have zero results yet).
_played_any = sched[sched["result"].notna()]
LATEST_COMPLETED = int(_played_any["season"].max())
LATEST_SCHEDULED = int(sched["season"].max())
if SEASON_MAX is None:
    SEASON_MAX = LATEST_COMPLETED
if TARGET_SEASON is None:
    TARGET_SEASON = LATEST_SCHEDULED

sched = sched[(sched["season"] >= SEASON_MIN) & (sched["season"] <= max(SEASON_MAX, TARGET_SEASON))].copy()

first_kickoff = (sched.groupby("season")["gameday"].min()
                 .rename("first_kickoff").reset_index())
SCHED_HASH = sha256_frame(sched[["game_id", "season", "week", "home_team", "away_team", "result"]])

print(f"source          : {SCHED_SOURCE}")
print(f"seasons in scope: {SEASON_MIN}–{SEASON_MAX}   (target/predict season: {TARGET_SEASON})")
print(f"REG games       : {len(sched):,}   played: {int(sched['result'].notna().sum()):,}")
print(f"schedule hash   : {SCHED_HASH[:16]}…")
print(first_kickoff.tail(4).to_string(index=False))

source          : snapshot:futures/data/schedules_snapshot.parquet
seasons in scope: 2002–2025   (target/predict season: 2026)
REG games       : 6,495   played: 6,223
schedule hash   : 6d97ff5b662015b1…
 season first_kickoff
   2023    2023-09-07
   2024    2024-09-05
   2025    2025-09-04
   2026    2026-09-09


### Interpreting the output

2026-08-03 run: **2002–2025 in scope, target 2026; 6,495 REG games, 6,223 played.** That 272-game
gap *is* the unplayed 2026 season — arithmetic confirmation no future results leaked in. The
per-season first kickoff is what makes "preseason" decidable in Section 5.

### What these tests guard

REG-only (playoffs would inflate wins), every game dated (a null date would silently fail the
point-in-time comparison), no duplicate `game_id`, exactly 32 franchises with no historical
abbreviation surviving, and a published schedule for the target season.

In [6]:
if RUN_TESTS:
    assert {"season", "week", "home_team", "away_team", "result", "gameday"} <= set(sched.columns)
    assert sched["season"].between(SEASON_MIN, max(SEASON_MAX, TARGET_SEASON)).all()
    assert (sched["game_type"] == "REG").all(), "non-REG games leaked into the schedule frame"
    assert sched["gameday"].notna().all(), "every scheduled game needs a date (point-in-time gate)"
    assert not sched["game_id"].duplicated().any(), "duplicate game_id in schedule"
    # franchise normalization is total: no historical abbr survives
    _fr = set(sched["home_franchise"]) | set(sched["away_franchise"])
    assert not (_fr & set(FRANCHISE_MAP)), f"unmapped historical abbreviations remain: {_fr & set(FRANCHISE_MAP)}"
    assert len(_fr) == 32, f"expected 32 franchises, got {len(_fr)}"
    # the predict season must have a published schedule and no results yet, or be a completed season
    _t = sched[sched["season"] == TARGET_SEASON]
    assert len(_t) > 0, f"no schedule rows for TARGET_SEASON={TARGET_SEASON}"
    print(f"✓ Section 3 tests passed | {len(sched):,} REG games, 32 franchises, "
          f"{sched['season'].nunique()} seasons, target {TARGET_SEASON} "
          f"({int(_t['result'].notna().sum())}/{len(_t)} played)")

✓ Section 3 tests passed | 6,495 REG games, 32 franchises, 25 seasons, target 2026 (0/272 played)


### Reading the test result

`target 2026 (0/272 played)` is the leakage check made visible: full opponent list, zero outcomes.
Does **not** prove the schedule data is *correct*, only structurally sound.

## Section 4 — Outcome table (the target)

One row per team-season. Target is **`wins_half_ties`** (§2.1: a tie grades as half a win); strict
`wins` rides along for re-grading. `games_played` is computed per team, not assumed — 16-game and
17-game eras, plus BUF/CIN at 16 in 2022. A season is `complete` only when every scheduled game has
a result, which is how the upcoming season excludes itself.

In [7]:
_played = sched[sched["result"].notna()].copy()

_home = pd.DataFrame({
    "season": _played["season"], "team": _played["home_team"], "franchise": _played["home_franchise"],
    "margin": _played["result"], "pf": _played["home_score"], "pa": _played["away_score"],
})
_away = pd.DataFrame({
    "season": _played["season"], "team": _played["away_team"], "franchise": _played["away_franchise"],
    "margin": -_played["result"], "pf": _played["away_score"], "pa": _played["home_score"],
})
_tg = pd.concat([_home, _away], ignore_index=True)
_tg["win"]  = (_tg["margin"] > 0).astype(float)
_tg["loss"] = (_tg["margin"] < 0).astype(float)
_tg["tie"]  = (_tg["margin"] == 0).astype(float)

outcomes = (_tg.groupby(["season", "franchise"])
            .agg(games_played=("win", "size"), wins=("win", "sum"), losses=("loss", "sum"),
                 ties=("tie", "sum"), points_for=("pf", "sum"), points_against=("pa", "sum"))
            .reset_index())
outcomes["wins_half_ties"] = outcomes["wins"] + 0.5 * outcomes["ties"]
outcomes["point_diff"] = outcomes["points_for"] - outcomes["points_against"]
outcomes["win_pct"] = outcomes["wins_half_ties"] / outcomes["games_played"]

# scheduled (not just played) games per team-season -> completeness flag
_sh = sched[["season", "home_franchise"]].rename(columns={"home_franchise": "franchise"})
_sa = sched[["season", "away_franchise"]].rename(columns={"away_franchise": "franchise"})
scheduled = (pd.concat([_sh, _sa], ignore_index=True)
             .groupby(["season", "franchise"]).size().rename("games_scheduled").reset_index())
outcomes = outcomes.merge(scheduled, on=["season", "franchise"], how="right")
outcomes[["games_played", "wins", "losses", "ties", "wins_half_ties"]] = \
    outcomes[["games_played", "wins", "losses", "ties", "wins_half_ties"]].fillna(0.0)
outcomes["games_played"] = outcomes["games_played"].astype(int)

season_status = (outcomes.groupby("season")
                 .agg(teams=("franchise", "nunique"),
                      games_played=("games_played", "sum"),
                      games_scheduled=("games_scheduled", "sum"))
                 .reset_index())
season_status["complete"] = season_status["games_played"] == season_status["games_scheduled"]
COMPLETE_SEASONS = sorted(season_status.loc[season_status["complete"], "season"].astype(int))
outcomes_complete = outcomes[outcomes["season"].isin(COMPLETE_SEASONS)].reset_index(drop=True)
OUTCOME_HASH = sha256_frame(outcomes_complete[["season", "franchise", "games_played", "wins_half_ties"]])

print(f"complete seasons : {len(COMPLETE_SEASONS)}  ({COMPLETE_SEASONS[0]}–{COMPLETE_SEASONS[-1]})")
print(f"team-seasons     : {len(outcomes_complete):,}")
print(f"outcome hash     : {OUTCOME_HASH[:16]}…")
print(season_status.tail(5).to_string(index=False))
print("\nseason lengths observed:",
      dict(outcomes_complete.groupby("games_played").size().items()))

complete seasons : 24  (2002–2025)
team-seasons     : 768
outcome hash     : b60d2ccd466215bf…
 season  teams  games_played  games_scheduled  complete
   2022     32           542              542      True
   2023     32           544              544      True
   2024     32           544              544      True
   2025     32           544              544      True
   2026     32             0              544     False

season lengths observed: {16: 610, 17: 158}


### Interpreting the output

**24 complete seasons (2002–2025), 768 team-seasons** = 24 × 32, itself a check. Season lengths run
`{16: 610, 17: 158}`, so any metric assuming a fixed denominator is wrong on a third of the panel.
The outcome hash pins this table for `01` to re-check.

### What these tests guard

This is **G4**. The strongest check is league wins conservation — `Σ wins_half_ties == games played`,
per season — which catches a dropped game, a double-count, a margin sign error and a mishandled tie
at once. Ties must also actually exist, or the half-win branch is untested code.

In [8]:
if RUN_TESTS:
    # --- G4: outcome-table integrity (PREREGISTRATION §5) ---
    for _s in COMPLETE_SEASONS:
        _o = outcomes_complete[outcomes_complete["season"] == _s]
        assert len(_o) == 32, f"{_s}: expected 32 team-seasons, got {len(_o)}"
        # league wins conservation: every played game awards exactly 1.0 (or 0.5+0.5 on a tie)
        _games = int(sched[(sched["season"] == _s) & sched["result"].notna()].shape[0])
        _sum = float(_o["wins_half_ties"].sum())
        assert abs(_sum - _games) < 1e-9, f"{_s}: Σwins_half_ties={_sum} != games played={_games}"
        # per-team record identity
        assert (_o["wins"] + _o["losses"] + _o["ties"] == _o["games_played"]).all(), f"{_s}: W+L+T != GP"
        assert (_o["games_played"] == _o["games_scheduled"]).all(), f"{_s}: marked complete but GP != GS"
    assert outcomes_complete["wins_half_ties"].between(0, outcomes_complete["games_played"]).all()
    assert outcomes_complete[["season", "franchise"]].duplicated().sum() == 0, "duplicate team-season rows"
    assert outcomes_complete[["wins", "losses", "ties", "points_for", "points_against"]].notna().all().all()
    # ties must actually exist somewhere in the panel, else the half-win convention is untested
    assert outcomes_complete["ties"].sum() > 0, "no ties found — verify the tie branch, not just the totals"
    # the incomplete / upcoming season must NOT be in the modelling universe
    assert TARGET_SEASON not in COMPLETE_SEASONS or TARGET_SEASON <= LATEST_COMPLETED
    print(f"✓ Section 4 tests passed | {len(COMPLETE_SEASONS)} complete seasons, "
          f"{len(outcomes_complete):,} team-seasons, wins conserved in every season, "
          f"{int(outcomes_complete['ties'].sum())} team-ties graded at 0.5")

✓ Section 4 tests passed | 24 complete seasons, 768 team-seasons, wins conserved in every season, 30 team-ties graded at 0.5


### Reading the test result

An exact identity checked 24 times, plus 30 team-ties proving the half-win path runs on real data.
Does **not** prove half-a-win is the right rule for whatever book supplied the lines — that's the
§2.1 assumption, which is why strict `wins` is carried.

## Section 5 — Market-line coverage audit (§2.2 + §10 Amendment 1)

**The load-bearing section.** Amendment 1 splits G3:

* **G3-B** — valid date + named `market_source`; `book` may be null → §7 gates **A and B only**.
* **G3-C** — frozen rule, **named book** + strictly pre-kickoff → the sole key to **gate C**.

**Kickoff-day clause (A1.3):** a date *on* Week 1 is admitted for Tier B only, on the source's own
closing statement; **exact timestamps are unavailable**. No date = not admitted. The strictly-dated
subset is computed separately for the mandatory A1.4 sensitivity.

In [9]:
LINE_REQUIRED_COLS = ["season", "team", "win_total_line", "price_over", "price_under",
                      "book", "as_of_date", "source"]
# Amendment 1 adds one required column on the archive path: the archive must name itself.
TIER_B_REQUIRED_COLS = LINE_REQUIRED_COLS + ["market_source"]
REQUIRED_COLS = TIER_B_REQUIRED_COLS if TIER_B_ARCHIVE else LINE_REQUIRED_COLS

# --- what we searched (reported whether or not anything is found) -------------------------
search_log = []
LINES_FILE = None
for _why, _p in _lines_candidates:
    _exists = _p.exists()
    search_log.append({"candidate": _why, "path": str(_p), "exists": bool(_exists)})
    if _exists and LINES_FILE is None:
        LINES_FILE = _p

# a wider sweep, purely informational: anything on disk that looks like a futures/win-total file
_sweep = []
for _pat in ("*win*total*", "*futures*", "*season*total*"):
    for _d in (DATA_DIR, REPO / "betting" / "data", REPO / "futures"):
        if _d.is_dir():
            _sweep += [p for p in _d.glob(_pat) if p.is_file()]
SWEEP_HITS = sorted({str(p.relative_to(REPO).as_posix()) for p in _sweep})

lines = pd.DataFrame(columns=REQUIRED_COLS)
LINES_SCHEMA_ERRORS, LINES_FILE_HASH = [], None

if LINES_FILE is not None:
    _raw = pd.read_csv(LINES_FILE)
    LINES_FILE_HASH = sha256_file(LINES_FILE)
    _missing = [c for c in REQUIRED_COLS if c not in _raw.columns]
    if _missing:
        LINES_SCHEMA_ERRORS.append(f"missing required columns: {_missing}")
    else:
        lines = _raw[REQUIRED_COLS].copy()
        lines["season"] = pd.to_numeric(lines["season"], errors="coerce").astype("Int64")
        lines["win_total_line"] = pd.to_numeric(lines["win_total_line"], errors="coerce")
        for _c in ("price_over", "price_under"):
            lines[_c] = pd.to_numeric(lines[_c], errors="coerce")
        lines["as_of_date"] = pd.to_datetime(lines["as_of_date"], errors="coerce")
        lines["franchise"] = lines["team"].astype(str).str.upper().replace(FRANCHISE_MAP)

# --- per-row validity under §2.2 as amended ------------------------------------------------
if len(lines):
    lines = lines.merge(first_kickoff, on="season", how="left")
    lines["has_price"] = lines["price_over"].notna() & lines["price_under"].notna()
    lines["has_book"] = lines["book"].notna() & (lines["book"].astype(str).str.strip() != "")
    lines["has_market_source"] = (lines["market_source"].notna() &
                                  (lines["market_source"].astype(str).str.strip() != "")) \
        if "market_source" in lines.columns else pd.Series(False, index=lines.index)
    lines["has_as_of"] = lines["as_of_date"].notna() & lines["first_kickoff"].notna()
    # point-in-time status is DERIVED here from the dates, never read from the file
    lines["strictly_before"] = lines["has_as_of"] & (lines["as_of_date"] < lines["first_kickoff"])
    lines["same_day"] = lines["has_as_of"] & (lines["as_of_date"] == lines["first_kickoff"])
    lines["after_kickoff"] = lines["has_as_of"] & (lines["as_of_date"] > lines["first_kickoff"])
    # A1.3: same-day admitted on the archive path only; after-kickoff never admitted
    lines["is_preseason"] = lines["strictly_before"] | (lines["same_day"] & bool(TIER_B_ARCHIVE))
    lines["known_team"] = lines["franchise"].isin(set(outcomes["franchise"]))
    _base = (lines["win_total_line"].notna() & lines["has_price"] &
             lines["known_team"] & lines["is_preseason"])
    lines["valid_b"] = _base & (lines["has_market_source"] if TIER_B_ARCHIVE else lines["has_book"])
    # G3-C is the FROZEN rule: named book AND strictly before kickoff. Unchanged by the amendment.
    lines["valid_c"] = (lines["win_total_line"].notna() & lines["has_price"] & lines["known_team"] &
                        lines["strictly_before"] & lines["has_book"])
    lines["valid"] = lines["valid_b"]
    lines["is_integer_line"] = lines["win_total_line"].notna() & (lines["win_total_line"] % 1 == 0)

    def _coverage(mask):
        _v = lines[mask]
        if not len(_v):
            return pd.DataFrame(columns=["season", "teams_covered", "rows", "books", "earliest_as_of",
                                         "latest_as_of", "integer_lines", "pct_integer",
                                         "outcome_available", "pit_status"])
        _c = (_v.groupby("season")
              .agg(teams_covered=("franchise", "nunique"), rows=("franchise", "size"),
                   books=("book", "nunique"), earliest_as_of=("as_of_date", "min"),
                   latest_as_of=("as_of_date", "max"), integer_lines=("is_integer_line", "sum"),
                   n_strict=("strictly_before", "sum"))
              .reset_index())
        _c["pct_integer"] = 100 * _c["integer_lines"] / _c["rows"]
        _c["outcome_available"] = _c["season"].isin(COMPLETE_SEASONS)
        _c["pit_status"] = np.where(_c["n_strict"] == _c["rows"],
                                    "strictly_before_week1", "same_day_as_week1_kickoff")
        return _c

    coverage = _coverage(lines["valid_b"])
    coverage_strict = _coverage(lines["valid_b"] & lines["strictly_before"])
else:
    coverage = coverage_strict = pd.DataFrame(
        columns=["season", "teams_covered", "rows", "books", "earliest_as_of", "latest_as_of",
                 "integer_lines", "pct_integer", "outcome_available", "pit_status"])

# --- gates G1-G3 --------------------------------------------------------------------------
G2_MIN_TEAMS, G1_MIN_SEASONS = 28, 8


def _counted(cov):
    return cov[(cov["teams_covered"] >= G2_MIN_TEAMS) & cov["outcome_available"]] if len(cov) else cov


_cnt = _counted(coverage)
USABLE_SEASONS = sorted(_cnt["season"].astype(int)) if len(_cnt) else []
USABLE_SEASONS_STRICT = sorted(_counted(coverage_strict)["season"].astype(int)) if len(coverage_strict) else []

G1 = {"name": "G1 >= 8 seasons with usable preseason lines", "threshold": G1_MIN_SEASONS,
      "observed": len(USABLE_SEASONS), "passed": len(USABLE_SEASONS) >= G1_MIN_SEASONS}
G2 = {"name": f"G2 >= {G2_MIN_TEAMS}/32 teams covered in every counted season",
      "threshold": G2_MIN_TEAMS,
      "observed": int(_cnt["teams_covered"].min()) if len(_cnt) else 0,
      "passed": bool(len(_cnt)) and bool((_cnt["teams_covered"] >= G2_MIN_TEAMS).all())}
_nb_rows = int(lines["valid_b"].sum()) if len(lines) else 0
_nc_rows = int(lines["valid_c"].sum()) if len(lines) else 0
G3B = {"name": "G3-B every counted row is point-in-time with a named market_source (Amendment 1; "
               "Tier B only, book may be null)",
       "threshold": "all rows", "observed": f"{_nb_rows} valid of {len(lines)} rows",
       "passed": bool(_nb_rows > 0 and not LINES_SCHEMA_ERRORS and bool(TIER_B_ARCHIVE))}
G3C = {"name": "G3-C every counted row is strictly pre-kickoff with a NAMED BOOK (frozen §5; "
               "required for §7 gate C)",
       "threshold": "all rows", "observed": f"{_nc_rows} valid of {len(lines)} rows",
       "passed": bool(_nc_rows > 0 and not LINES_SCHEMA_ERRORS)}

print("searched for a win-total line file:")
for _r in search_log:
    print(f"  [{'FOUND' if _r['exists'] else 'absent'}] {_r['candidate']:<26} {_r['path']}")
print(f"\nwider sweep for futures-looking files: {SWEEP_HITS or 'none'}")
if LINES_SCHEMA_ERRORS:
    print(f"\nSCHEMA ERRORS in {LINES_FILE}: {LINES_SCHEMA_ERRORS}")
print(f"\nline rows loaded          : {len(lines):,}")
print(f"valid on the ARCHIVE path : {_nb_rows:,}  (G3-B, Tier B only)")
print(f"valid on the BOOK path    : {_nc_rows:,}  (G3-C, required for gate C)")
print(f"usable seasons (admitted) : {USABLE_SEASONS or 'none'}")
print(f"usable seasons (strict)   : {USABLE_SEASONS_STRICT or 'none'}   <- A1.4 sensitivity")
if len(coverage):
    print("\n" + coverage.drop(columns=["integer_lines"]).to_string(index=False))

searched for a win-total line file:
  [FOUND] futures/data/win_totals.csv C:\Users\josep\Desktop\random_stuff\cowork_OS\JoSchoAnalytics\futures\data\win_totals.csv

wider sweep for futures-looking files: ['futures/01_acquire_win_totals.ipynb', 'futures/data/win_totals.csv']

line rows loaded          : 352
valid on the ARCHIVE path : 352  (G3-B, Tier B only)
valid on the BOOK path    : 0  (G3-C, required for gate C)
usable seasons (admitted) : [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2024, 2025]
usable seasons (strict)   : [2014, 2015, 2016, 2017, 2018]   <- A1.4 sensitivity

 season  teams_covered  rows  books earliest_as_of latest_as_of  n_strict  pct_integer  outcome_available                pit_status
   2014             32    32      0     2014-09-01   2014-09-01        32       53.125               True     strictly_before_week1
   2015             32    32      0     2015-09-07   2015-09-07        32       34.375               True     strictly_before_week1
   2016

### Interpreting the output

2026-08-03: **352 valid on the archive path, 0 on the book path** — the whole reason gate C is
unreachable, visible as a number rather than a caveat. **11 usable seasons** (needs 8); the strict
subset is **5** (2014–2018). `books` reads 0 everywhere — the attribution lives in `market_source`,
and the audit keeps those two facts separate.

### What these tests guard

Mostly against the audit being generous to itself: nothing after kickoff and nothing dateless is
admitted on either path; **G3-C can never count a row without a named book or without strict
priority**; with the amendment off, both paths must coincide exactly; and zero rows may never pass
vacuously.

In [10]:
if RUN_TESTS:
    assert set(LINE_REQUIRED_COLS) == {"season", "team", "win_total_line", "price_over",
                                       "price_under", "book", "as_of_date", "source"}, \
        "the frozen §2.2 schema is the data contract — changing it needs an amendment"
    assert set(REQUIRED_COLS) >= set(LINE_REQUIRED_COLS), "the amendment may only ADD requirements"
    if TIER_B_ARCHIVE:
        assert "market_source" in REQUIRED_COLS, "A1.1 requires a named market_source on the archive path"
    assert all(isinstance(r["exists"], bool) for r in search_log) and len(search_log) >= 1
    assert all(s in COMPLETE_SEASONS for s in USABLE_SEASONS), \
        "a season with no settled outcome can never be a usable backtest season"
    assert set(USABLE_SEASONS_STRICT) <= set(USABLE_SEASONS), "strict subset must be a subset"
    for _g in (G1, G2, G3B, G3C):
        assert isinstance(_g["passed"], bool), f"gate {_g['name']} did not decide"
    if len(lines):
        # A1.3: nothing after kickoff is EVER admitted, on either path
        assert not (lines["valid_b"] & lines["after_kickoff"]).any(), "an in-season row was admitted"
        assert not (lines["valid_c"] & lines["after_kickoff"]).any()
        # G3-C is the frozen rule and may never be satisfied without a named book
        assert not (lines["valid_c"] & ~lines["has_book"]).any(), \
            "G3-C counted a row with no named book — that is the gate that guards §7 gate C"
        assert not (lines["valid_c"] & ~lines["strictly_before"]).any(), \
            "G3-C counted a same-day row — the frozen rule requires strict priority"
        # the amendment must not be able to admit a dateless row
        assert not (lines["valid_b"] & ~lines["has_as_of"]).any(), "a dateless row was admitted"
        # and with the amendment OFF the two paths must coincide
        if not TIER_B_ARCHIVE:
            assert (lines["valid_b"] == lines["valid_c"]).all(), \
                "with TIER_B_ARCHIVE=False the gate must reproduce the frozen behaviour exactly"
    else:
        assert not any(g["passed"] for g in (G1, G2, G3B, G3C)), \
            "no line rows loaded, so no line gate may report a pass"
    print(f"✓ Section 5 tests passed | G1={G1['passed']} ({G1['observed']} seasons) G2={G2['passed']} "
          f"G3-B={G3B['passed']} ({_nb_rows} rows) G3-C={G3C['passed']} ({_nc_rows} rows) | "
          f"strict subset {len(USABLE_SEASONS_STRICT)} seasons")

✓ Section 5 tests passed | G1=True (11 seasons) G2=True G3-B=True (352 rows) G3-C=False (0 rows) | strict subset 5 seasons


### Reading the test result

`G3-B=True (352 rows) G3-C=False (0 rows)` is the amendment in one line — admissible for accuracy,
inadmissible for anything priced. Amendment 1 widens one path and is mechanically incapable of
widening the other. Does **not** prove the archived numbers are accurate transcriptions.

## Section 6 — Baseline availability

§4 requires baselines computable on the same rows the model is scored on. Measures coverage of B1
(persistence, rescaled across the 16→17 boundary) and B2 (league mean). Fits nothing.

In [11]:
_prior = outcomes_complete[["season", "franchise", "wins_half_ties", "games_played", "point_diff"]].copy()
_prior["season"] = _prior["season"] + 1
_prior = _prior.rename(columns={"wins_half_ties": "prior_wins", "games_played": "prior_games",
                                "point_diff": "prior_point_diff"})

baselines = outcomes_complete.merge(_prior, on=["season", "franchise"], how="left")
# B1 persistence, rescaled when the season length changed (16 -> 17 games)
baselines["b1_persistence"] = baselines["prior_wins"] * (baselines["games_played"] / baselines["prior_games"])
baselines["b2_league_mean"] = baselines["games_played"] / 2.0

b1_cov = (baselines.groupby("season")
          .agg(rows=("franchise", "size"), b1_available=("b1_persistence", lambda s: int(s.notna().sum())))
          .reset_index())
b1_cov["pct"] = 100 * b1_cov["b1_available"] / b1_cov["rows"]

_scored = baselines.dropna(subset=["b1_persistence"])
B1_MAE = float((_scored["wins_half_ties"] - _scored["b1_persistence"]).abs().mean())
B2_MAE = float((baselines["wins_half_ties"] - baselines["b2_league_mean"]).abs().mean())

print(f"B1 persistence available on {len(_scored):,}/{len(baselines):,} team-seasons "
      f"({100*len(_scored)/len(baselines):.1f}%)")
print(f"in-sample descriptive MAE (NOT a result — no fold structure): "
      f"B1={B1_MAE:.3f}  B2(league mean)={B2_MAE:.3f} wins")
print(b1_cov.head(3).to_string(index=False), "\n…\n", b1_cov.tail(3).to_string(index=False))

B1 persistence available on 736/768 team-seasons (95.8%)
in-sample descriptive MAE (NOT a result — no fold structure): B1=2.895  B2(league mean)=2.557 wins
 season  rows  b1_available   pct
   2002    32             0   0.0
   2003    32            32 100.0
   2004    32            32 100.0 
…
  season  rows  b1_available   pct
   2023    32            32 100.0
   2024    32            32 100.0
   2025    32            32 100.0


### Interpreting the output

Persistence covers **736/768**; the 32 gaps are 2002, which has no prior. The MAEs point against
intuition: **persistence 2.895 vs league mean 2.557** — predicting 8.5 for everyone beats predicting
last year. So B2 is the honest floor, gate A's threshold is a low bar, and the market was always the
comparison that mattered. In-sample and fold-free — not results.

### What these tests guard

No coverage holes after the first season (a hole means a franchise failed to join across a
relocation), and the 16→17 rescale **actually moved 2021's values** — a rescale that silently no-ops
is the classic "code is there but does nothing" defect.

In [12]:
if RUN_TESTS:
    # the first in-scope season can have no prior; every later season must be fully covered
    _later = b1_cov[b1_cov["season"] > min(COMPLETE_SEASONS)]
    assert (_later["pct"] == 100).all(), \
        f"persistence baseline has holes after the first season:\n{_later[_later['pct'] < 100]}"
    assert b1_cov.loc[b1_cov["season"] == min(COMPLETE_SEASONS), "b1_available"].iloc[0] == 0
    # the 16->17 rescale must actually move 2021 values off the raw prior
    _b21 = baselines[(baselines["season"] == 2021) & baselines["prior_wins"].notna()]
    if len(_b21):
        assert not np.allclose(_b21["b1_persistence"], _b21["prior_wins"]), \
            "2021 persistence was not rescaled for the 17th game"
    assert 0 < B1_MAE < 6 and 0 < B2_MAE < 6, "baseline MAEs are implausible — check the join"
    # B2 must equal half the season length exactly
    assert np.allclose(baselines["b2_league_mean"] * 2, baselines["games_played"])
    print(f"✓ Section 6 tests passed | B1 covers {len(_scored):,} rows, "
          f"rescale applied at the 2021 boundary, B1 MAE {B1_MAE:.3f} B2 MAE {B2_MAE:.3f}")

✓ Section 6 tests passed | B1 covers 736 rows, rescale applied at the 2021 boundary, B1 MAE 2.895 B2 MAE 2.557


### Reading the test result

Confirms the baselines are computable on the rows they'll be used on. Does **not** settle whether
persistence deserves to be a baseline given the league mean beats it — B3 (shrunk persistence) is
the sensible middle and is fitted per fold in `02`.

## Section 7 — Pre-Week-1 feature availability

Classifies every candidate family **before any feature is built**, so availability can't be decided
retroactively by whatever helps: **AVAILABLE** / **CONDITIONAL** (needs a dated snapshot we don't
own) / **UNAVAILABLE** (backfilled or revised — using it is a leak). Season *S*'s *opponents* are
preseason information; season *S*'s *results* are not.

In [13]:
FEATURE_AVAILABILITY = pd.DataFrame([
    ("prior_season_record",      "load_schedules (season S-1)", "AVAILABLE",
     "settled before S; wins/losses/ties/point differential"),
    ("prior_season_pythag",      "load_schedules (season S-1)", "AVAILABLE",
     "points for/against of S-1 only"),
    ("multi_year_record",        "load_schedules (S-3..S-1)",   "AVAILABLE",
     "decayed multi-season form; all settled"),
    ("prior_season_pbp_epa",     "load_pbp (season S-1)",       "AVAILABLE",
     "off/def EPA per play from S-1; final and unrevised at kickoff of S"),
    ("schedule_strength_S",      "load_schedules (season S)",   "AVAILABLE",
     "opponents are published before Week 1 — the schedule itself is preseason information; "
     "opponent STRENGTH must be measured from S-1, never from S results"),
    ("coach_identity",           "load_schedules (season S)",   "AVAILABLE",
     "coach of record per game; preseason coach = coach of week 1"),
    ("coach_prior_winpct",       "load_schedules (< S)",        "AVAILABLE",
     "career win% through S-1 only"),
    ("qb_identity_week1",        "load_schedules (season S)",   "CONDITIONAL",
     "week-1 starter is known preseason in reality, but the nflverse field is populated after "
     "the game is played — a preseason snapshot is required to use it honestly"),
    ("roster_continuity",        "load_rosters (season S)",     "CONDITIONAL",
     "the roster table is revised in place through the season; needs a dated preseason snapshot"),
    ("depth_chart_rank",         "load_depth_charts",           "UNAVAILABLE",
     "coverage ends at 2024 in this stack (the documented train-present/deploy-absent trap that "
     "forced Amendment 1 in fantasy/projections) — excluded by construction"),
    ("free_agency_transactions", "external",                    "CONDITIONAL",
     "no dated transaction feed is owned by this repo"),
    ("preseason_injuries",       "load_injuries (season S)",    "CONDITIONAL",
     "the injury report is a weekly in-season feed; week-1 report exists but is published in "
     "game week, and prior weeks are not preseason state"),
    ("season_S_results",         "load_schedules (season S)",   "UNAVAILABLE",
     "the target — any use is the leak this audit exists to prevent"),
    ("season_S_pbp",             "load_pbp (season S)",         "UNAVAILABLE",
     "same-season play-by-play is post-kickoff by definition"),
], columns=["feature_family", "source", "verdict", "note"])

AVAILABLE_FAMILIES = sorted(FEATURE_AVAILABILITY.loc[
    FEATURE_AVAILABILITY["verdict"] == "AVAILABLE", "feature_family"])

# Verify (not merely assert) the checkable AVAILABLE claims on the schedule frame itself:
# for the predict season the opponent list must be fully known while results are absent.
_t = sched[sched["season"] == TARGET_SEASON]
TARGET_SCHEDULE_KNOWN = bool(len(_t) > 0 and _t[["home_franchise", "away_franchise"]].notna().all().all())
TARGET_RESULTS_PRESENT = int(_t["result"].notna().sum())

print(FEATURE_AVAILABILITY.to_string(index=False))
print(f"\nAVAILABLE families: {len(AVAILABLE_FAMILIES)}  "
      f"CONDITIONAL: {(FEATURE_AVAILABILITY['verdict'] == 'CONDITIONAL').sum()}  "
      f"UNAVAILABLE: {(FEATURE_AVAILABILITY['verdict'] == 'UNAVAILABLE').sum()}")
print(f"target season {TARGET_SEASON}: schedule known={TARGET_SCHEDULE_KNOWN}, "
      f"results present={TARGET_RESULTS_PRESENT}")

          feature_family                      source     verdict                                                                                                                                                             note
     prior_season_record load_schedules (season S-1)   AVAILABLE                                                                                                            settled before S; wins/losses/ties/point differential
     prior_season_pythag load_schedules (season S-1)   AVAILABLE                                                                                                                                   points for/against of S-1 only
       multi_year_record   load_schedules (S-3..S-1)   AVAILABLE                                                                                                                           decayed multi-season form; all settled
    prior_season_pbp_epa       load_pbp (season S-1)   AVAILABLE                                

### Interpreting the output

**7 AVAILABLE, 4 CONDITIONAL, 3 UNAVAILABLE.** The AVAILABLE seven are all prior-season or
published-schedule quantities — a thin feature space, which is itself informative. The CONDITIONAL
four fail structurally: the upstream feed is revised in place, so today's value isn't what was
knowable then. `depth_chart_rank` is UNAVAILABLE by construction (coverage ends 2024 — the trap that
forced Amendment 1 in `fantasy/projections`).

### What these tests guard

A closed verdict vocabulary; `season_S_results`, `season_S_pbp` and `depth_chart_rank` pinned
UNAVAILABLE by name (reclassifying one means deleting an assertion); every verdict carries a reason;
and — highest value — **the prior-season frame was shifted the correct direction**. Subtracting
instead of adding would join each season to its own future.

In [14]:
if RUN_TESTS:
    assert set(FEATURE_AVAILABILITY["verdict"]) <= {"AVAILABLE", "CONDITIONAL", "UNAVAILABLE"}
    assert not FEATURE_AVAILABILITY["feature_family"].duplicated().any()
    # the target and same-season PBP must never be classified usable
    for _f in ("season_S_results", "season_S_pbp", "depth_chart_rank"):
        _v = FEATURE_AVAILABILITY.loc[FEATURE_AVAILABILITY["feature_family"] == _f, "verdict"].iloc[0]
        assert _v == "UNAVAILABLE", f"{_f} must be UNAVAILABLE, got {_v}"
    assert "season_S_results" not in AVAILABLE_FAMILIES and "season_S_pbp" not in AVAILABLE_FAMILIES
    # every family carries a stated reason — an unexplained verdict is not a decision
    assert (FEATURE_AVAILABILITY["note"].str.len() > 20).all()
    # the AVAILABLE claim about the published schedule is verified, not assumed
    assert TARGET_SCHEDULE_KNOWN, f"{TARGET_SEASON} opponents are not fully published"
    # leakage guard: a prior-season join must never see the season it predicts
    _pj = _prior[_prior["season"] == TARGET_SEASON]
    assert (_pj["prior_games"] > 0).all() if len(_pj) else True
    assert (_prior["season"] > outcomes_complete["season"].min()).all(), \
        "the prior-season frame was shifted the wrong way — that is a direct leak"
    print(f"✓ Section 7 tests passed | {len(FEATURE_AVAILABILITY)} families classified, "
          f"{len(AVAILABLE_FAMILIES)} AVAILABLE, target-season results withheld "
          f"({TARGET_RESULTS_PRESENT} present)")

✓ Section 7 tests passed | 14 families classified, 7 AVAILABLE, target-season results withheld (0 present)


### Reading the test result

14 families classified, 7 AVAILABLE, target season holding 0 results. Does **not** prove `01`'s
implementations will honour the classification — which is why it's written into the artifact rather
than living only in this notebook's prose.

## Section 8 — Verdict and artifact

`GO` needs G3-C; `GO-TIER-B` needs G3-B and leaves `tier_c_open` False; anything else is `NO-GO`.
Both fold sets are frozen here — before any model exists — so neither the headline nor the A1.4
sensitivity can be redefined after a result is seen. The artifact also records that exact closing
timestamps are unavailable.

In [15]:
G4 = {"name": "G4 outcome-table integrity (32 teams/season, wins conserved, GP matches schedule)",
      "threshold": "all seasons", "observed": f"{len(COMPLETE_SEASONS)} complete seasons verified",
      "passed": True}   # Section 4's test cell fails the notebook before this line if not

GATES = [G1, G2, G3B, G3C, G4]
_core = G1["passed"] and G2["passed"] and G4["passed"]
if _core and G3C["passed"]:
    VERDICT = "GO"
elif _core and G3B["passed"] and TIER_B_ARCHIVE:
    VERDICT = "GO-TIER-B"
else:
    VERDICT = "NO-GO"

TIER_C_OPEN = (VERDICT == "GO")          # only a named book opens the priced ladder
TIER = {"GO": "A+B+C", "GO-TIER-B": "A+B", "NO-GO": "none"}[VERDICT]

FOLDS = USABLE_SEASONS[1:] if VERDICT.startswith("GO") else []
FOLDS_STRICT = USABLE_SEASONS_STRICT[1:] if VERDICT.startswith("GO") else []
MIN_FOLDS, MIN_EVAL_ROWS = 5, 160
_eval_rows = int(coverage.loc[coverage["season"].isin(FOLDS), "teams_covered"].sum()) if FOLDS else 0
_eval_rows_strict = int(coverage_strict.loc[coverage_strict["season"].isin(FOLDS_STRICT),
                                            "teams_covered"].sum()) if FOLDS_STRICT else 0
UNDERPOWERED = VERDICT.startswith("GO") and (len(FOLDS) < MIN_FOLDS or _eval_rows < MIN_EVAL_ROWS)
UNDERPOWERED_STRICT = len(FOLDS_STRICT) < MIN_FOLDS or _eval_rows_strict < MIN_EVAL_ROWS

audit = {
    "verdict": VERDICT,
    "tier_available": TIER,
    "tier_c_open": bool(TIER_C_OPEN),
    "underpowered": bool(UNDERPOWERED),
    "gates": GATES,
    "amendment_1": {
        "active": bool(TIER_B_ARCHIVE),
        "reference": "futures/PREREGISTRATION.md §10 Amendment 1 (accepted 2026-08-03)",
        "admits": "archived market consensus with a null `book`, for §7 gates A and B only",
        "market_source_is_not_a_sportsbook": True,
        "exact_closing_timestamps_available": False,
        "kickoff_day_clause_used": bool(len(USABLE_SEASONS) > len(USABLE_SEASONS_STRICT)),
        "locked": ["§7 gate C", "sides", "probability against a posted line", "confidence tiers",
                   "EV", "profitability", "bet/edge/lock/value/play language"],
    },
    "folds": {"test_seasons": FOLDS, "n_folds": len(FOLDS), "eval_rows": _eval_rows,
              "min_folds": MIN_FOLDS, "min_eval_rows": MIN_EVAL_ROWS,
              "rule": "expanding-season: train on all seasons < T with line+outcome, test on T"},
    "folds_strict_sensitivity": {
        "test_seasons": FOLDS_STRICT, "n_folds": len(FOLDS_STRICT), "eval_rows": _eval_rows_strict,
        "underpowered": bool(UNDERPOWERED_STRICT),
        "rule": "A1.4 — strictly-dated seasons only; MANDATORY second reporting of every headline; "
                "sensitivity only, never the headline (below §3's five-fold minimum)"},
    "target": {"column": "wins_half_ties",
               "settlement_assumption": "tie = half a win (book convention, PREREGISTRATION §2.1)",
               "strict_wins_carried": True},
    "outcomes": {"season_min": int(SEASON_MIN), "season_max": int(SEASON_MAX),
                 "complete_seasons": [int(s) for s in COMPLETE_SEASONS],
                 "team_seasons": int(len(outcomes_complete)),
                 "hash": OUTCOME_HASH},
    "predict_season": {"season": int(TARGET_SEASON),
                       "schedule_published": bool(TARGET_SCHEDULE_KNOWN),
                       "results_present": int(TARGET_RESULTS_PRESENT)},
    "lines": {"file": _rel(LINES_FILE) if LINES_FILE else None,
              "file_sha256": LINES_FILE_HASH,
              "required_schema": REQUIRED_COLS,
              "frozen_schema": LINE_REQUIRED_COLS,
              "schema_errors": LINES_SCHEMA_ERRORS,
              "rows_loaded": int(len(lines)),
              "rows_valid": int(_nb_rows),
              "rows_valid_book_path": int(_nc_rows),
              "usable_seasons": [int(s) for s in USABLE_SEASONS],
              "usable_seasons_strict": [int(s) for s in USABLE_SEASONS_STRICT],
              "searched": search_log,
              "sweep_hits": SWEEP_HITS,
              "pct_integer_lines": (float(coverage["pct_integer"].mean()) if len(coverage) else None),
              "books": sorted(lines.loc[lines["valid_b"], "book"].dropna().astype(str).unique().tolist())
                       if len(lines) else [],
              "market_sources": sorted(lines.loc[lines["valid_b"], "market_source"].dropna()
                                       .astype(str).unique().tolist())
                                if len(lines) and "market_source" in lines.columns else []},
    "baselines": {"b1_persistence_rows": int(len(_scored)),
                  "b1_in_sample_mae": B1_MAE, "b2_in_sample_mae": B2_MAE,
                  "note": "in-sample descriptive only; no fold structure, not a result"},
    "feature_availability": FEATURE_AVAILABILITY.to_dict(orient="records"),
    "schedule": {"source": SCHED_SOURCE, "library_version": SCHED_LIB_VERSION,
                 "snapshot": _rel(SNAPSHOT_PATH),
                 "hash": SCHED_HASH, "reg_games": int(len(sched))},
    "provenance": PROVENANCE,
}

if WRITE_ARTIFACTS:
    AUDIT_PATH.write_text(json.dumps(audit, indent=2, default=str), encoding="utf-8")

print("=" * 78)
print(f"  DATA AUDIT VERDICT: {VERDICT}     (§7 tiers available: {TIER}, gate C open: {TIER_C_OPEN})")
print("=" * 78)
for g in GATES:
    print(f"  [{'PASS' if g['passed'] else 'FAIL'}] {g['name']}")
    print(f"         observed: {g['observed']}   threshold: {g['threshold']}")
print("-" * 78)
if VERDICT == "NO-GO":
    print("  No usable point-in-time preseason win-total lines. Per §5 the subproject STOPS HERE.")
    print("  Substituting current-season lines or reconstructing historical ones is forbidden.")
else:
    print(f"  folds (test seasons) : {FOLDS}  |  eval rows: {_eval_rows}")
    print(f"  A1.4 strict subset   : {FOLDS_STRICT}  |  eval rows: {_eval_rows_strict}"
          f"{'  [UNDERPOWERED — sensitivity only]' if UNDERPOWERED_STRICT else ''}")
    if UNDERPOWERED:
        print("  ⚠ UNDERPOWERED per §3 — results may be reported descriptively but open no gate.")
    if VERDICT == "GO-TIER-B":
        print("  TIER B ONLY (Amendment 1). Permitted: projection quality, and accuracy against an")
        print("  ARCHIVED MARKET CONSENSUS of unattributed sportsbook origin, in aggregate.")
        print("  LOCKED: gate C, sides, probability vs a posted line, confidence, EV, profitability.")
        print("  Exact closing timestamps are UNAVAILABLE — this travels with every result.")
    print("  → 01_build_dataset.ipynb may run.")
print("=" * 78)
print(f"  artifact: {AUDIT_PATH if WRITE_ARTIFACTS else '(not written — WRITE_ARTIFACTS=False)'}")

  DATA AUDIT VERDICT: GO-TIER-B     (§7 tiers available: A+B, gate C open: False)
  [PASS] G1 >= 8 seasons with usable preseason lines
         observed: 11   threshold: 8
  [PASS] G2 >= 28/32 teams covered in every counted season
         observed: 32   threshold: 28
  [PASS] G3-B every counted row is point-in-time with a named market_source (Amendment 1; Tier B only, book may be null)
         observed: 352 valid of 352 rows   threshold: all rows
  [FAIL] G3-C every counted row is strictly pre-kickoff with a NAMED BOOK (frozen §5; required for §7 gate C)
         observed: 0 valid of 352 rows   threshold: all rows
  [PASS] G4 outcome-table integrity (32 teams/season, wins conserved, GP matches schedule)
         observed: 24 complete seasons verified   threshold: all seasons
------------------------------------------------------------------------------
  folds (test seasons) : [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2024, 2025]  |  eval rows: 320
  A1.4 strict subset   : [20

### Interpreting the output

**`GO-TIER-B`** — G1 (11 seasons), G2 (32/32), G3-B and G4 pass; **G3-C fails at 0 rows**.
Headline folds **10 seasons / 320 rows**; A1.4 strict subset **4 folds / 128 rows**, flagged
UNDERPOWERED. The sensitivity can contradict the headline but never become it, and if they disagree
in sign the strict subset governs. The Tier-B block printed under the verdict is the claim license.

### What these tests guard

The verdict follows from the gates on the right path: **`GO` is defined to require G3-C**, so no
setting of `TIER_B_ARCHIVE` can produce one; `tier_c_open` tracks G3-C and nothing else; a
`GO-TIER-B` run must carry no named book while having a named market source; NO-GO is a hard stop
checked physically against downstream artifacts on disk; and the JSON round-trips.

In [16]:
if RUN_TESTS:
    assert VERDICT in ("GO", "GO-TIER-B", "NO-GO")
    # the verdict must follow from the gates, on the right path
    assert (VERDICT == "GO") == (_core and G3C["passed"]), "GO must require the frozen book gate G3-C"
    assert (VERDICT == "GO-TIER-B") == (_core and not G3C["passed"] and G3B["passed"] and bool(TIER_B_ARCHIVE))
    # A1.5: the priced ladder is opened by G3-C and by nothing else
    assert TIER_C_OPEN == G3C["passed"] or not _core, "tier_c_open must track G3-C"
    if VERDICT == "GO-TIER-B":
        assert TIER_C_OPEN is False, "Tier B must never open gate C"
        assert TIER == "A+B"
        assert audit["amendment_1"]["exact_closing_timestamps_available"] is False
        assert not audit["lines"]["books"], "a Tier-B run must carry no named book"
        assert audit["lines"]["market_sources"], "A1.1 requires the archive to name itself"
    if VERDICT == "NO-GO":
        assert FOLDS == [] and FOLDS_STRICT == [], "a NO-GO run must not publish a fold set"
        for _forbidden in ("futures_predictions.csv", "artifacts/model_metadata.json"):
            assert not (FUTURES / _forbidden).exists(), \
                f"NO-GO but {_forbidden} exists — a downstream notebook ran past the gate"
    else:
        assert len(FOLDS) >= 1 and min(FOLDS) > min(USABLE_SEASONS), \
            "the earliest usable season must be training-only"
        assert all(s in COMPLETE_SEASONS for s in FOLDS)
        # A1.4: the sensitivity fold set must exist and be a strict subset
        assert set(FOLDS_STRICT) <= set(FOLDS), "the strict sensitivity must be a subset of the headline"
    if WRITE_ARTIFACTS:
        _back = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
        assert _back["verdict"] == VERDICT and _back["tier_c_open"] == TIER_C_OPEN
        assert _back["folds"]["test_seasons"] == FOLDS
        assert _back["folds_strict_sensitivity"]["test_seasons"] == FOLDS_STRICT
        assert _back["target"]["column"] == "wins_half_ties"
        assert _back["provenance"]["as_of_date"] == PROVENANCE["as_of_date"]
        assert _back["outcomes"]["hash"] == OUTCOME_HASH
        assert _back["schedule"]["hash"] == SCHED_HASH
        assert _back["amendment_1"]["active"] == bool(TIER_B_ARCHIVE)
        assert len(_back["feature_availability"]) == len(FEATURE_AVAILABILITY)
    print(f"✓ Section 8 tests passed | verdict={VERDICT} tier={TIER} gate_C_open={TIER_C_OPEN} "
          f"folds={FOLDS} strict={FOLDS_STRICT} artifact={'written' if WRITE_ARTIFACTS else 'skipped'}")

✓ Section 8 tests passed | verdict=GO-TIER-B tier=A+B gate_C_open=False folds=[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2024, 2025] strict=[2015, 2016, 2017, 2018] artifact=written


### Reading the test result

Verdict, tier, gate-C state and both fold sets in one line. Those assertions make it mechanically
impossible for the amendment to unlock the priced ladder. Does **not** prove Tier B is *worth*
running — `02` finds that out, and §7 gate A still has to pass before anything reaches the site.

## Conclusion and next steps

**Verdict: `GO-TIER-B`** (2026-08-03) on 352 rows / 11 seasons from Covers. G1 11≥8, G2 32/32,
G3-B pass, **G3-C fail (0 rows — no named book)**, G4 pass. `tier_c_open: False`.

**Folds frozen:** headline 10 test seasons (320 rows); A1.4 strict-subset sensitivity 4 seasons
(128 rows, underpowered — sensitivity only, never the headline).

**Licensed:** projection quality, and accuracy vs an *archived market consensus of unattributed
sportsbook origin*, in aggregate, reported twice. **Locked:** gate C, sides, probability vs a posted
line, confidence, EV, profitability, and *bet/edge/lock/value/play*. Exact closing timestamps are
unavailable.

**Next:** run `01_build_dataset.ipynb` (it reads the frozen folds, never recomputes them). For
anything priced, collect timestamped named-book 2026 lines — the only route to G3-C.